# Webinar 2: Data Preprocessing — Track 2: Text Pipeline
### Dataset: 20 Newsgroups (Real Internet Forum Discussions — sci.med vs alt.atheism)
### Algorithms: LinearSVC (SVM) vs Baseline SGD Classifier

This notebook covers the complete NLP text preprocessing workflow on real discussion posts:
1. **Text Cleaning**: Removing email artifacts, quotation headers, special characters, and stopwords.
2. **TF-IDF Vectorization**: Extracting unigram + bigram representations with sublinear TF scaling.
3. **Numerical Feature Engineering**: Extracting text length, uppercase shouting ratio, punctuation density.
4. **Imbalance Handling**: Addressing class imbalance using SMOTE on text embeddings.
5. **Model Evaluation**: Comparing raw Bag-of-Words SGD Classifier vs Cleaned TF-IDF **LinearSVC**.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.text import (
    clean_text,
    batch_clean_texts,
    TFIDFProcessor,
    extract_numerical_text_features,
    run_text_pipeline
)

print('Text NLP modules imported successfully!')

Text NLP modules imported successfully!


## Step 1: Inspect Real Internet Forum Posts
Observing real-world noise: quotation tags, email fragments, contractions, and typos.

In [2]:
df_text = pd.read_csv('../data/text/newsgroups_raw.csv')
print(f'Total documents: {len(df_text)}')
df_text.head()

Total documents: 1780


,doc_id,raw_text,category,target
0,DOC_0001,In article <1993Apr28.173600.21703@organpipe.u...,alt.atheism,1
1,DOC_0002,In article <1993Apr5.023044.19580@ultb.isc.rit...,sci.med,0
2,DOC_0003,In article <1993Apr2.155057.808@batman.bmd.trw...,sci.med,0
3,DOC_0004,Greeting\n\nI am starting work on a project wh...,alt.atheism,1
4,DOC_0005,Andrew Newell (TAN102@psuvm.psu.edu) wrote:\n:...,sci.med,0


## Step 2: Cleaning & Text Normalization

In [3]:
sample_raw = df_text['raw_text'].iloc[0]
sample_cleaned = clean_text(sample_raw)
print(f'BEFORE CLEANING (first 250 chars):\n{sample_raw[:250]}...\n')
print(f'AFTER CLEANING (first 250 chars):\n{sample_cleaned[:250]}...')

BEFORE CLEANING (first 250 chars):
In article <1993Apr28.173600.21703@organpipe.uug.arizona.edu> ame_0123@bigdog.engr.arizona.edu (Terrance J. Dishongh) writes:
>Greeting
>
>I am starting work on a project where I am trying to make strain gages
>bond to bone in vivo or a period of sev...

AFTER CLEANING (first 250 chars):
article terrance dishongh writes greeting starting work project trying make strain gages bond bone vivo period several months currently using hydroxyapaptite back gages tried bonding gages bone apart two application methods seem much else literature ...


## Step 3: TF-IDF Vectorization with N-Grams

In [4]:
cleaned_texts = batch_clean_texts(df_text['raw_text'])
tfidf = TFIDFProcessor(max_features=500, ngram_range=(1, 2), sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(cleaned_texts)
print(f'TF-IDF Matrix Shape: {tfidf_matrix.shape}')
top_kw = tfidf.get_top_keywords(cleaned_texts, top_n=10)
top_kw

TF-IDF Matrix Shape: (1780, 500)


,term,mean_tfidf_score
0,writes,0.0558
1,article,0.0514
2,would,0.0486
3,can,0.0460
4,one,0.0455
5,know,0.0367
6,people,0.0357
7,like,0.0332
8,think,0.0329
9,god,0.0296


## Step 4: Linguistic Numerical Feature Engineering

In [5]:
num_feats = extract_numerical_text_features(df_text['raw_text'])
num_feats.head()

,char_count,word_count,avg_word_length,uppercase_count,uppercase_ratio,exclamation_count,question_count,digit_count,lexical_diversity,pos_keyword_count,neg_keyword_count,lexicon_polarity_score
0,1292,198,5.48,35,0.0271,0,0,30,0.6818,0,0,0
1,1260,228,4.38,63,0.0500,0,1,20,0.5965,2,0,2
2,5936,981,4.93,103,0.0174,1,10,24,0.4424,10,2,8
3,530,95,4.52,11,0.0208,0,0,0,0.7053,0,0,0
4,6022,1073,4.56,150,0.0249,1,4,9,0.3868,1,0,1


## Step 5: Full NLP Pipeline Execution (LinearSVC vs SGD)

In [6]:
text_results = run_text_pipeline('../data/text/newsgroups_raw.csv')

from src.evaluation.comparison import generate_modality_comparison
comp_df = generate_modality_comparison(text_results['baseline_metrics'], text_results['preprocessed_metrics'], 'Text (20 Newsgroups)')
comp_df

C:\Users\Roger\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


  Text | Baseline: SGD Classifier -> Preprocessed: LinearSVC (Calibrated)


  Top TF-IDF keywords: writes, article, would, can, one


,Modality,Metric,Baseline (Before),Preprocessed (After),Delta,Relative Improvement
0,Text (20 Newsgroups),Accuracy,0.9034,0.9596,+0.0562,+6.2%
1,Text (20 Newsgroups),Precision,0.9143,0.9597,+0.0454,+5.0%
2,Text (20 Newsgroups),Recall,0.9106,0.9675,+0.0569,+6.3%
3,Text (20 Newsgroups),F1-Score (Macro),0.9023,0.9591,+0.0567,+6.3%
4,Text (20 Newsgroups),F1-Score (Minority),0.9124,0.9636,+0.0511,+5.6%
5,Text (20 Newsgroups),ROC-AUC,0.9037,0.9877,+0.0840,+9.3%
